# BanRep — TES / IBR / DTF semanales

Construye un único CSV con tres tasas BanRep agregadas a frecuencia semanal (cierre de viernes), listo para unir con los datos consolidados de Finagro/SFC.

**Series**
- **TES** — `Tasa de interés Cero Cupón, Títulos de Tesorería (TES), pesos - 1 año` (suameca serie [640001](https://suameca.banrep.gov.co/estadisticas-economicas/informacionSerie/640001/deuda_publica_tasas_cero_cupon_tes)).
- **IBR** — `Indicador Bancario de Referencia (IBR) overnight, nominal`.
- **DTF** — `Tasa de Depósitos a Término Fijo (DTF) a 90 días, semanal`.

Los CSV de origen se descargan a mano desde suameca y se suben (i) localmente a `Fuentes de Datos/` y/o (ii) a Google Drive (luego pegar los IDs en la celda de configuración).

**Salida**: `tasas_banrep_semanal.csv` con columnas `fecha, tes_1y, ibr_overnight, dtf_90d` — un renglón por viernes.

Convenciones de columnas IBR/DTF: ver [`Creación_DF.ipynb`](Creaci%C3%B3n_DF.ipynb). Semántica de viernes: ver [`../Análisis/Consolidación_Finagro_y_SFC.ipynb`](../An%C3%A1lisis/Consolidaci%C3%B3n_Finagro_y_SFC.ipynb).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

## Configuración (entorno + rutas)

Detecta Colab vs local. En Colab, descarga los tres CSV con `gdown` usando los IDs de Drive (rellénalos una vez). En local, lee directamente desde el mismo directorio del notebook (`Fuentes de Datos/`).

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Reemplaza estos IDs con los de tus archivos en Drive.
    GDRIVE_IDS = {
        'tes': '<TES_CSV_DRIVE_ID>',
        'ibr': '<IBR_CSV_DRIVE_ID>',
        'dtf': '<DTF_CSV_DRIVE_ID>',
    }

    SRC_DIR = Path('/content')
    OUTPUT_PATH = Path('/content/drive/MyDrive/finagro/tasas_banrep_semanal.csv')

    for name, gid in GDRIVE_IDS.items():
        out = SRC_DIR / f'{name}.csv'
        if not out.exists():
            os.system(f"gdown --id '{gid}' --output '{out}'")
else:
    SRC_DIR = Path('.')
    OUTPUT_PATH = Path('tasas_banrep_semanal.csv')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f'IN_COLAB={IN_COLAB}  SRC_DIR={SRC_DIR}  OUTPUT_PATH={OUTPUT_PATH}')

## Carga de los CSV de BanRep

Los tres CSV de suameca comparten el mismo formato: separador `;`, decimal `,`, columna de fecha `Periodo(MMM DD, AAAA)` con valores `YYYY/MM/DD`. `load_banrep_csv` selecciona la columna de valor cuyo nombre contenga el patrón dado y la renombra a `canonical_name`.

In [ ]:
def load_banrep_csv(path: Path, value_col_pattern: str, canonical_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=';', decimal=',')

    date_col = 'Periodo(MMM DD, AAAA)'
    if date_col not in df.columns:
        raise ValueError(f"{path.name}: no encuentro la columna de fecha {date_col!r}. Columnas: {list(df.columns)}")

    matches = [c for c in df.columns if value_col_pattern.lower() in c.lower()]
    if len(matches) == 0:
        raise ValueError(
            f"{path.name}: ninguna columna contiene {value_col_pattern!r}. "
            f"Columnas disponibles: {list(df.columns)}"
        )
    if len(matches) > 1:
        raise ValueError(
            f"{path.name}: el patrón {value_col_pattern!r} matchea {len(matches)} columnas: {matches}. "
            'Usa un patrón más específico.'
        )
    value_col = matches[0]

    out = df[[date_col, value_col]].rename(columns={date_col: 'fecha', value_col: canonical_name})
    out['fecha'] = pd.to_datetime(out['fecha'], format='%Y/%m/%d')
    out = out.dropna(subset=[canonical_name]).sort_values('fecha').reset_index(drop=True)
    return out

In [ ]:
tes = load_banrep_csv(SRC_DIR / 'tes.csv', value_col_pattern='pesos - 1 año', canonical_name='tes_1y')
ibr = load_banrep_csv(SRC_DIR / 'ibr.csv', value_col_pattern='overnight',      canonical_name='ibr_overnight')
dtf = load_banrep_csv(SRC_DIR / 'dtf.csv', value_col_pattern='DTF',            canonical_name='dtf_90d')

for name, df in [('TES 1y', tes), ('IBR overnight', ibr), ('DTF 90d', dtf)]:
    print(f'{name:>15}: {len(df):>6} filas, {df["fecha"].min().date()} → {df["fecha"].max().date()}')

## Agregación semanal (cierre de viernes)

Resample `W-FRI` con `.last()`: toma la última observación disponible de cada semana. Coincide con la lógica de "Agrupar por viernes" del notebook de consolidación. La DTF ya viene semanal — pasarla por el mismo paso solo alinea su índice con las otras.

In [ ]:
def to_weekly_friday(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    return (df.set_index('fecha')
              .resample('W-FRI')
              .last()
              .reset_index()
              .dropna(subset=[value_col]))

tes_w = to_weekly_friday(tes, 'tes_1y')
ibr_w = to_weekly_friday(ibr, 'ibr_overnight')
dtf_w = to_weekly_friday(dtf, 'dtf_90d')

## Outer-join sobre `fecha`

In [ ]:
merged = (tes_w.merge(ibr_w, on='fecha', how='outer')
               .merge(dtf_w, on='fecha', how='outer')
               .sort_values('fecha')
               .reset_index(drop=True))

print('Shape:', merged.shape)
print('Rango:', merged['fecha'].min().date(), '→', merged['fecha'].max().date())
print('\nNaNs por columna:')
print(merged.isna().sum())
print('\nDtypes:')
print(merged.dtypes)
print('\nPrimeras filas:')
print(merged.head())
print('\nÚltimas filas:')
print(merged.tail())

## Escritura del CSV

In [ ]:
merged.to_csv(OUTPUT_PATH, index=False)
print(f'Escrito: {OUTPUT_PATH}  ({OUTPUT_PATH.stat().st_size:,} bytes)')

## Re-correr semanalmente

1. Bajar los CSV actualizados desde suameca (TES serie 640001, IBR, DTF) y reemplazar `tes.csv` / `ibr.csv` / `dtf.csv` en `Fuentes de Datos/` (o re-subirlos a Drive y `gdown` los traerá si los borras de `/content`).
2. Correr todas las celdas. El CSV de salida se sobrescribe — la salida es determinista (mismo input → mismo CSV).

**Para extender** (más tenores de IBR o más maturities de TES): agregar más llamadas a `load_banrep_csv` con su patrón y nombre canónico, pasarlas por `to_weekly_friday`, y añadirlas a la cadena de `merge`.